# Build ROQ Basis for mlgw\_bns\_jax (GW170817)

This notebook builds Reduced Order Quadrature (ROQ) interpolants for the
**mlgw\_bns** waveform model using [JenpyROQ](https://github.com/bernuzzi/JenpyROQ).

Uses the **original NumPy-based** `mlgw_bns` model (no JAX), avoiding
LLVM JIT compilation memory issues on low-RAM machines.

The ROQ basis compresses the ~253k-point frequency grid down to O(1000) empirical
nodes, accelerating likelihood evaluations by ~100×.

**Target:** Google Colab / IGWN JupyterHub.  
**Runtime:** several hours (depending on hardware).

In [ ]:
import os, subprocess, sys, shutil

COLAB = "google.colab" in sys.modules
IGWN  = os.path.exists("/cvmfs/oasis.opensciencegrid.org")

REPO_DIR = "/content/mlgw_bns_jax" if COLAB else os.getcwd()

# ── Install dependencies ─────────────────────────────────────────────
if COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "h5py", "JenpyROQ", "scikit-learn",
    ])
    # Clone the repo (model + wrapper + config)
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call([
            "git", "clone", "--branch", "blackjax_ns_gw_pe", "--depth", "1",
            "https://github.com/jacopok/mlgw_bns.git", REPO_DIR,
        ])
    os.chdir(REPO_DIR)
    # Install mlgw_bns in-place
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
elif IGWN:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "JenpyROQ", "h5py",
    ])

# Verify working directory
if os.path.basename(os.getcwd()) != "mlgw_bns_jax":
    candidate = os.path.expanduser("~/mlgw_bns_jax")
    if os.path.isdir(candidate):
        os.chdir(candidate)

print(f"Working directory: {os.getcwd()}")
print("All required files found.")

In [ ]:
import os, logging, sys
import numpy as np

# Register the mlgw_bns wrapper (NumPy — no JAX needed for ROQ build)
import mlgw_bns_roq_wrapper  # side-effect: registers WfWrapper

from JenpyROQ.jenpyroq import JenpyROQ
from JenpyROQ.initialise import read_config
from JenpyROQ.parallel import initialize_serial_pool

print("JenpyROQ loaded, NumPy wrapper registered (no JAX).")

# Logging
logger = logging.getLogger("JenpyROQ")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s  %(message)s"))
    logger.addHandler(handler)

# Read configuration
CONFIG_FILE = "config_roq_mlgw_bns_jax_gw170817.ini"
OUT_DIR     = "./roq_basis_mlgw_bns_jax/"
os.makedirs(OUT_DIR, exist_ok=True)

config_pars, params_ranges, test_values = read_config(CONFIG_FILE, OUT_DIR, logger)

wf_cfg = config_pars["Waveform_and_parametrisation"]
fmin = wf_cfg['f-min']
fmax = wf_cfg['f-max']
seglen = wf_cfg['seglen']
print(f"\nFrequency: [{fmin}, {fmax}] Hz")
print(f"Segment length: {seglen} s  ->  df = {1/seglen:.4f} Hz")
print(f"Training ranges: {params_ranges}")

In [ ]:
from mlgw_bns_roq_wrapper import WfMLGWBNS

# Smoke test — full 253k grid, pure NumPy
wf = WfMLGWBNS("mlgw-bns-jax")
p_test = {
    "m1": 1.365, "m2": 1.365,
    "s1z": 0.0, "s2z": 0.0,
    "lambda1": 300.0, "lambda2": 300.0,
    "iota": 2.5, "phiref": 0.6,
}
hp_test, hc_test = wf.generate_waveform(p_test, 1.0/seglen, fmin, fmax, 10.0)
print(f"Waveform OK: hp shape={hp_test.shape}, max|hp|={np.max(np.abs(hp_test)):.3e}")
del hp_test, hc_test

## Build the ROQ basis

This is the main computation. It runs:

1. **Pre-selection** (corner basis + greedy selection of ~100 elements)
2. **Enrichment** (3 cycles with 10k / 50k / 100k training waveforms)

Each cycle generates random waveforms, projects them onto the current basis,
and adds the worst-represented waveform to the basis until the tolerance is met.

**Linear basis** (tolerance 1e-4): used in ⟨d|h⟩ inner products  
**Quadratic basis** (tolerance 1e-6): used in ⟨h|h⟩ inner products

In [ ]:
import time, gc

pool = initialize_serial_pool()

print("=" * 60)
print("Building ROQ basis for mlgw-bns (NumPy)")
print(f"  f = [{fmin}, {fmax}] Hz")
print(f"  seglen = {seglen} s")
print(f"  Output: {OUT_DIR}")
print("=" * 60)

t0 = time.time()

with pool as p:
    roq = JenpyROQ(config_pars, params_ranges, distance=10.0, pool=p)

    # ── Phase 1: LINEAR basis ──────────────────────────────────────
    print("\n--- Building LINEAR basis ---")
    data_lin = roq.run("lin")
    n_lin = len(data_lin["lin_emp_nodes"])
    print(f"Linear basis: {n_lin} elements")

    # Free linear arrays before quadratic build.
    # All results are already saved to disk by JenpyROQ
    # (roq_basis_mlgw_bns_jax/ROQ_data/linear/*.npy)
    del data_lin
    gc.collect()
    print("(linear data freed from RAM — saved on disk)")

    # ── Phase 2: QUADRATIC basis ───────────────────────────────────
    print("\n--- Building QUADRATIC basis ---")
    data_qua = roq.run("qua")
    n_qua = len(data_qua["qua_emp_nodes"])
    print(f"Quadratic basis: {n_qua} elements")

elapsed = time.time() - t0

# Summary
f_full = np.arange(fmin, fmax + 1.0 / seglen, 1.0 / seglen)
n_full = len(f_full)

sep = "=" * 60
print(f"\n{sep}")
print(f"  Full frequency grid : {n_full} points")
print(f"  Linear basis        : {n_lin} elements  ({n_full / n_lin:.0f}x reduction)")
print(f"  Quadratic basis     : {n_qua} elements  ({n_full / n_qua:.0f}x reduction)")
print(f"  Elapsed time        : {elapsed/3600:.1f} hours")
print(f"  Output directory    : {OUT_DIR}")
print(sep)

## Validation

Check the ROQ representation error on random test waveforms.

In [ ]:
import matplotlib.pyplot as plt

# ── Reload linear basis from disk (freed from RAM during build) ───────
roq_lin_dir = os.path.join(OUT_DIR, "ROQ_data", "linear")
B_lin     = np.load(os.path.join(roq_lin_dir, "basis_interpolant_linear.npy"))
nodes_lin = np.load(os.path.join(roq_lin_dir, "empirical_nodes_linear.npy"))

roq_qua_dir = os.path.join(OUT_DIR, "ROQ_data", "quadratic")
B_qua     = np.load(os.path.join(roq_qua_dir, "basis_interpolant_quadratic.npy"))
nodes_qua = np.load(os.path.join(roq_qua_dir, "empirical_nodes_quadratic.npy"))

print(f"Linear  ROQ nodes: {len(nodes_lin)} ({n_full/len(nodes_lin):.0f}x speedup)")
print(f"Quadratic ROQ nodes: {len(nodes_qua)} ({n_full/len(nodes_qua):.0f}x speedup)")

# ── Test: generate a waveform and compare full vs ROQ ─────────────────
from mlgw_bns_roq_wrapper import WfMLGWBNS
wf = WfMLGWBNS("mlgw-bns-jax")

deltaF = 1.0 / seglen
# GW170817-like parameters
mc, q = 1.197, 1.0
eta = q / (1 + q)**2
m_total = mc / eta**0.6
m1 = m_total * q / (1 + q)
m2 = m_total / (1 + q)
p_wf = {"m1": m1, "m2": m2, "s1z": 0.0, "s2z": 0.0,
        "lambda1": 300.0, "lambda2": 300.0, "iota": 2.5, "phiref": 0.6}

hp_full, hc_full = wf.generate_waveform(p_wf, deltaF, fmin, fmax, 10.0)

# Normalise for comparison
from JenpyROQ.linear_algebra import normalise_vector, scalar_product
hp_norm = normalise_vector(hp_full, deltaF)

# Linear ROQ reconstruction
hp_roq_lin = np.dot(B_lin, hp_norm[nodes_lin])

# Residual
residual = hp_norm - hp_roq_lin
eie = scalar_product(residual, residual, deltaF)
tol_lin = config_pars["ROQ"]["tolerance-lin"]
print(f"\nLinear interpolation error: {float(np.real(eie)):.2e}  (tolerance: {tol_lin})")

freq = np.arange(fmin, fmax + deltaF, deltaF)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={"height_ratios": [3, 1]})

ax = axes[0]
ax.plot(freq, np.real(hp_norm), lw=0.8, alpha=0.7, label="Full")
ax.plot(freq, np.real(hp_roq_lin), lw=0.5, ls="--", label="ROQ")
ax.scatter(freq[nodes_lin], np.real(hp_norm)[nodes_lin],
           s=8, c="red", zorder=5, label=f"Empirical nodes ({len(nodes_lin)})")
ax.set_ylabel(r"$\Re[\tilde{h}_+]$ (normalised)")
ax.set_title("Linear ROQ vs Full waveform")
ax.legend()

ax = axes[1]
ax.plot(freq, np.real(residual), lw=0.5, color="darkred")
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Residual")
ax.set_title(f"Representation error = {float(np.real(eie)):.2e}")

fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "roq_validation_linear.png"), dpi=150)
plt.show()
print("Validation plot saved.")

In [ ]:
import os

print("ROQ output files:")
print("=" * 60)
for root, dirs, files in os.walk(os.path.join(OUT_DIR, "ROQ_data")):
    level = root.replace(OUT_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in sorted(files):
        fpath = os.path.join(root, file)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"{subindent}{file}  ({size_mb:.1f} MB)")

## Download the ROQ basis

If running on Colab, download the output directory as a zip file.

In [ ]:
import shutil

zip_name = "roq_basis_mlgw_bns_jax"
roq_dir  = "./roq_basis_mlgw_bns_jax"

shutil.make_archive(zip_name, "zip", ".", "roq_basis_mlgw_bns_jax")
print(f"Created {zip_name}.zip ({os.path.getsize(zip_name + '.zip') / 1e6:.1f} MB)")

if COLAB:
    from google.colab import files
    files.download(f"{zip_name}.zip")
    print("Download started.")
else:
    print(f"Zip ready at: {os.path.abspath(zip_name + '.zip')}")